In [ ]:
# SPDX-FileCopyrightText: 2026 Mario Gemoll
# SPDX-License-Identifier: 0BSD

import os
import subprocess


def is_correct_repo() -> bool:
    try:
        result = subprocess.run(
            ["git", "remote", "get-url", "origin"],
            capture_output=True,
            text=True,
            check=True,
        )
        remote_url = result.stdout.strip()
        return remote_url in [
            "https://github.com/mariogemoll/reinforcement-learning.git",
            "git@github.com:mariogemoll/reinforcement-learning.git",
        ]
    except (subprocess.CalledProcessError, FileNotFoundError):
        return False


if not is_correct_repo():
    !git clone https://github.com/mariogemoll/reinforcement-learning.git

if not os.getcwd().endswith("reinforcement-learning/py"):
    %cd reinforcement-learning/py

In [ ]:
import jax
import jax.numpy as jnp
from tqdm.notebook import trange
from pendulum_pg import make_value_training_step, plot_training

In [ ]:
seed = 0
n_iter = 4000
num_rollouts = 32
max_steps = 200
lr_policy = 3e-4
value_lr_mult = 10
gamma = 0.99
chunk_size = 100

In [ ]:
make_carry, training_step = make_value_training_step(
    num_rollouts, max_steps, lr_policy, value_lr_mult, gamma
)

carry = make_carry(seed)
keys = jax.random.split(jax.random.PRNGKey(seed), n_iter)
n_chunks = n_iter // chunk_size
keys_chunked = keys.reshape(n_chunks, chunk_size, *keys.shape[1:])


def run_chunk(carry, keys):
    return jax.lax.scan(training_step, carry, keys)


jit_chunk = jax.jit(run_chunk)

losses_list, returns_list = [], []
pbar = trange(n_chunks, desc="Compiling...", leave=True)
for i in pbar:
    carry, (chunk_losses, chunk_returns) = jit_chunk(carry, keys_chunked[i])
    _ = chunk_returns.block_until_ready()
    losses_list.append(chunk_losses)
    returns_list.append(chunk_returns)
    pbar.set_description("Training")
    pbar.set_postfix(mean_return=f"{float(jnp.mean(chunk_returns)):.1f}")

losses = jnp.concatenate(losses_list)
returns = jnp.concatenate(returns_list)
plot_training(losses, returns, "Pendulum: Full Model (seed 0)")